# Shard-map viewer

This notebook demonstrates and gives **verification instructions** for the
`zagg.viz` shard-map viewer (issue
[#38](https://github.com/englacial/zagg/issues/38)).

The viewer has two layers:

- **Headless render core** (`zagg.viz.render_shardmap` and friends) -- pure
  Python, no browser/widget stack. It turns a `ShardMap` (and an optional
  `Catalog` / STAC-geoparquet file) into WGS84 GeoJSON `FeatureCollection`
  dicts: shard outlines, granule footprints, and viewport-clipped grid cells.
- **ipyleaflet wrapper** (`zagg.viz.show_shardmap`) -- builds an interactive
  map from those collections: a basemap, the shard-outline layer, a toggleable
  granule-footprint layer, and a grid layer that only draws when you zoom in
  far enough (the "grid-on-zoom" gate -- never a global graticule).

## Install

The headless core needs only the zagg core deps (`shapely` + the grid
backends). The interactive map needs the optional `viz` extra:

```bash
pip install zagg[viz]   # or: uv sync --extra viz
```

Everything below uses a tiny **synthetic** ShardMap and catalog, so the
notebook runs with no network, S3, or Earthdata credentials.

## 1. Build a small synthetic ShardMap and Catalog

We hand-build a `ShardMap` over a 4x4-chunk rectilinear grid (UTM 18N, near
Washington DC) with three populated shards, plus a two-granule STAC `Catalog`.
This mirrors the test fixtures in `tests/test_viz.py`. In real use you would
instead load a saved shard map and catalog produced by `python -m zagg.catalog`
(see [quickstart](https://github.com/englacial/zagg/blob/main/docs/quickstart.md)):

```python
from zagg.catalog.shardmap import ShardMap
from zagg.catalog.sources import Catalog
sm = ShardMap.from_json("shardmap_ATL06_cycle22.json")
cat = Catalog.from_geoparquet("catalog_ATL06_cycle22.parquet")
```

In [ ]:
import numpy as np
import pyarrow as pa
import stac_geoparquet.arrow as sga

from zagg.catalog.shardmap import ShardMap
from zagg.catalog.sources import Catalog
from zagg.grids import HealpixGrid, RectilinearGrid

# A 10 km x 10 km rectilinear grid at 10 m, chunked 250x250 -> one chunk == one
# shard. Three shards are populated with synthetic granule references.
grid = RectilinearGrid(
    "EPSG:32618", 10, [359400, 4300740, 369400, 4310740], [250, 250]
)
shard_keys = [0, 1, 5]
granules = [
    [{"id": "Ga", "s3": "s3://b/a.h5", "https": "https://h/a.h5"}],
    [{"id": "Gb", "s3": "s3://b/b.h5", "https": "https://h/b.h5"}],
    [
        {"id": "Gb", "s3": "s3://b/b.h5", "https": "https://h/b.h5"},
        {"id": "Gc", "s3": "s3://b/c.h5", "https": "https://h/c.h5"},
    ],
]
shardmap = ShardMap(grid.signature(), shard_keys, granules, {"backend": "demo"})
shardmap.shard_keys

In [ ]:
# A tiny STAC catalog of two granule footprints over the same area.
def _stac_item(gid, lon0, lon1, lat0=38.85, lat1=38.93):
    ring = [[lon0, lat0], [lon1, lat0], [lon1, lat1], [lon0, lat1], [lon0, lat0]]
    return {
        "type": "Feature", "stac_version": "1.0.0", "id": gid,
        "geometry": {"type": "Polygon", "coordinates": [ring]},
        "bbox": [lon0, lat0, lon1, lat1],
        "properties": {"datetime": "2025-06-01T00:00:00Z"},
        "collection": "DEMO", "stac_extensions": [], "links": [],
        "assets": {
            "data": {"href": f"https://h/{gid}.h5", "roles": ["data"]},
            "data_s3": {"href": f"s3://b/{gid}.h5", "roles": ["data"]},
        },
    }


items = [_stac_item("Ga", -76.62, -76.57), _stac_item("Gb", -76.55, -76.50)]
catalog = Catalog(
    pa.table(sga.parse_stac_items_to_arrow(items)), {"collection": "DEMO"}
)
len(list(catalog.granule_records()))

## 2. Headless render core: GeoJSON FeatureCollections

`render_shardmap` assembles every layer into one dict of GeoJSON
`FeatureCollection`s. Pass a `catalog` to include granule footprints and a
`bbox` to include the viewport-clipped grid cells. Each value is plain,
JSON-serializable GeoJSON -- no widgets required, so this is the part covered
by the unit tests.

In [ ]:
from zagg.viz import render_shardmap

layers = render_shardmap(shardmap, catalog)
{name: (fc and len(fc["features"])) for name, fc in layers.items()}

In [ ]:
# One feature per shard, with the shard key and its granule count. The grid is
# rebuilt from the map's own `grid_signature` -- the map is self-describing.
shards_fc = layers["shards"]
feat = shards_fc["features"][0]
print("geometry type:", feat["geometry"]["type"])
print("properties:", feat["properties"])
[f["properties"] for f in shards_fc["features"]]

In [ ]:
# One feature per granule footprint, decoded from the catalog.
[f["properties"]["id"] for f in layers["granules"]["features"]]

### Grid-on-zoom gate

`viewport_cells` draws shard-order cell outlines clipped to a viewport, but
**only** when `<= max_shards` shards intersect the viewport. Zoom in (a tight
bbox) and the grid appears; zoom out (a global bbox) and it returns an empty
collection rather than a globe-spanning graticule. The interactive map wires
this to the live `bounds`, so the grid blinks on as you zoom in.

In [ ]:
from zagg.viz import viewport_cells
from zagg.viz.shardmap import grid_from_signature

# Tight viewport over a single shard -> gate open, cells drawn.
fp = grid_from_signature(shardmap.grid_signature).shard_footprint(0)
lon0, lat0, lon1, lat1 = fp.bounds
zoomed_in = viewport_cells(shardmap, (lon0, lat0, lon1, lat1), max_shards=4)

# Global viewport -> too many shards visible -> gate closed, empty.
zoomed_out = viewport_cells(shardmap, (-180, -90, 180, 90), max_shards=1)

print("zoomed-in cells:", len(zoomed_in["features"]))
print("zoomed-out cells (gated):", len(zoomed_out["features"]))

### Antimeridian handling

HEALPix shards near +-180 deg are split into a `MultiPolygon` so they don't
render as a band wrapping the whole globe. Here a HEALPix parent cell that
straddles the seam comes back as a `MultiPolygon`, each part confined to one
hemisphere.

In [ ]:
from zagg.viz import shard_outlines

hp = HealpixGrid(2, 6, layout="fullsphere")
key = int(
    hp.coverage(
        [(np.array([0.0, 1, 1, 0, 0]), np.array([179.0, 179, 180, 180, 179]))]
    )[0]
)
sm_hp = ShardMap(hp.signature(), [key], [[{"id": "G", "s3": "s", "https": "h"}]], {})
geom = shard_outlines(sm_hp)["features"][0]["geometry"]
geom["type"]

## 3. Interactive map (ipyleaflet)

`show_shardmap` builds an `ipyleaflet.Map` from a **saved** ShardMap (and an
optional catalog). We round-trip the synthetic objects to disk first, since the
viewer accepts either in-memory objects or file paths.

**This cell needs `zagg[viz]` and is best run in JupyterLab** -- the widget
renders as a live, pannable/zoomable map there. Under headless `nbconvert` it
constructs the `Map` object (so the cell still executes) but won't show the
interactive tiles. See the verification checklist below for what to look for.

In [ ]:
import tempfile
from pathlib import Path

from zagg.viz import show_shardmap

tmp = Path(tempfile.mkdtemp())
sm_path = tmp / "shardmap.json"
cat_path = tmp / "catalog.parquet"
shardmap.to_json(str(sm_path))
catalog.to_geoparquet(str(cat_path))

# Construct the interactive map. In JupyterLab the returned Map renders inline.
m = show_shardmap(str(sm_path), catalog=str(cat_path), zoom=12)
print("layers:", [getattr(layer, "name", type(layer).__name__) for layer in m.layers])
m

## 4. Manual in-browser verification

Run this notebook in **JupyterLab** with the `viz` extra installed
(`pip install zagg[viz]`), execute the cells top-to-bottom, and confirm in the
map rendered by section 3:

1. **Basemap** -- an OpenStreetMap context map renders and pans/zooms.
2. **Shard outlines** -- the blue `shards` polygons (one per shard key) sit
   over the data area; hovering/clicking exposes their `shard_key` /
   `n_granules` properties.
3. **Granule footprints toggle** -- open the layer control (top-right) and
   toggle the red `granule footprints` layer on/off; the granule polygons
   appear/disappear.
4. **Grid-on-zoom** -- when zoomed out, the `grid (shard cells)` layer is empty
   (no global graticule). Zoom in until only a few shards fill the viewport and
   the grey shard-cell outlines appear; zoom back out and they vanish. This is
   the `max_shards` gate driven off the map's live `bounds`.

To verify the **headless** core without a browser, run the test suite:

```bash
uv run pytest tests/test_viz.py -v
```

For a real dataset, replace the synthetic objects in section 1 with a shard map
and catalog built via `python -m zagg.catalog` (see the quickstart) and pass
their paths to `show_shardmap`.